In [3]:
import cv2
import mediapipe as mp

# --- Step A: Choose one image path ---
image_path = "data/0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg.rf.4c570a281b5311eda118678acd2d6b7d.jpg"

# --- Step B: Load image ---
img = cv2.imread(image_path)
print("Image loaded:", img is not None)

# --- Step C: Prepare MediaPipe ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)

# --- Step D: Extract landmarks ---
result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

if result.multi_hand_landmarks:
    hand = result.multi_hand_landmarks[0]
    landmarks = []
    for lm in hand.landmark:
        landmarks.extend([lm.x, lm.y, lm.z])

    print("Total values:", len(landmarks))
    print("First 10 values:", landmarks[:10])
else:
    print("No hand detected.")


Image loaded: True
Total values: 63
First 10 values: [0.34647297859191895, 0.296077162027359, 1.623496643787803e-07, 0.3687577247619629, 0.29832136631011963, -0.0021665042731910944, 0.3874441385269165, 0.2846815586090088, -0.0021927764173597097, 0.3997296094894409]


d:\sujal\dev\Machine learning projects\sign lang detector\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


### now we do this to all the images and store the output with image and what their iamge mean in the file called keypoints

In [4]:
import pandas as pd

df = pd.read_csv("data/_classes.csv")
label_colums =['bye','hello','no','please','sorry','thankyou','yes']
def get_label_from_row(row):
    for col in label_colums:
        if row[col] == 1:
            return col
    return None
# lets test this shit out
for i in range(5):
    print(df.loc[i, "filename"], "→", get_label_from_row(df.loc[i]))


0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg.rf.3a6a224118a480f92f7afbdfb3fee1d8.jpg → bye
0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg.rf.5814b2f326b08cf68b50bdc43f60cb30.jpg → please
0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg.rf.36d70021f01622d21fd6ef117bc2487b.jpg → yes
0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg.rf.4c570a281b5311eda118678acd2d6b7d.jpg → hello
0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg.rf.09ffc7e8223de3fe1cc4c788c4fdc1f1.jpg → thankyou


In [5]:
# lets make this for every iamge and save this file in the dat folder

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=True, max_num_hands=1)

all_rows = []   # This will store all extracted data

for index, row in df.iterrows():

    filename = row["filename"]
    label = get_label_from_row(row)

    image_path = f"data/{filename}"
    img = cv2.imread(image_path)

    if img is None:
        print("Could not load:", image_path)
        continue

    result = hands.process(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))

    if not result.multi_hand_landmarks:
        print("No hand detected:", filename)
        continue

    hand = result.multi_hand_landmarks[0]

    keypoints = []
    for lm in hand.landmark:
        keypoints.extend([lm.x, lm.y, lm.z])

    # store: filename, all 63 values, label
    all_rows.append([filename] + keypoints + [label])

# Convert to DataFrame
columns = ["filename"] + [f"p{i}" for i in range(63)] + ["label"]
out_df = pd.DataFrame(all_rows, columns=columns)

out_df.to_csv("data/keypoints.csv", index=False)

print("Saved keypoints.csv with", len(out_df), "rows")

d:\sujal\dev\Machine learning projects\sign lang detector\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


No hand detected: 0009-dc05176e-6f42-4f5b-bf2a-6a6151608dab_jpg.rf.3d5daaf40464f6e607cd71b56450e7d0.jpg
No hand detected: 0017-34c0108f-954a-4498-90da-8e65e2a0e90a_jpg.rf.a33cdc567bbccceabb5f9aa0ad2a0514.jpg
No hand detected: 0020-91dd6b65-ed72-4271-976b-2ad1e5442c3c_jpg.rf.fd71972b36c0cbe27d5e230e9438b7fb.jpg
Saved keypoints.csv with 60 rows


In [6]:

df_kp = pd.read_csv("data/keypoints.csv")
df_kp.head()



,filename,p0,p1,p2,p3,p4,p5,p6,p7,p8,...,p54,p55,p56,p57,p58,p59,p60,p61,p62,label
0,0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg....,0.354855,0.377116,-2.079068e-07,0.381328,0.360973,0.000469,0.399949,0.335571,-0.000875,...,0.364259,0.269544,-0.016077,0.378706,0.279701,-0.014118,0.382285,0.295627,-0.010952,bye
1,0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg....,0.489402,0.658782,2.008355e-07,0.483480,0.612252,-0.008653,0.494541,0.570583,-0.016955,...,0.601380,0.632761,-0.038638,0.619705,0.621701,-0.039907,0.635699,0.613275,-0.039993,please
2,0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg....,0.421252,0.410285,-2.264598e-07,0.439472,0.411329,-0.007081,0.453712,0.415279,-0.016800,...,0.418672,0.396326,-0.025393,0.420461,0.409233,-0.022781,0.420964,0.414627,-0.019283,yes
3,0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg....,0.346473,0.296077,1.623497e-07,0.368758,0.298321,-0.002167,0.387444,0.284682,-0.002193,...,0.398149,0.219555,-0.005620,0.410957,0.209384,-0.004367,0.421394,0.201722,-0.003379,hello
4,0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg....,0.451162,0.550923,3.669905e-07,0.414900,0.518674,-0.011077,0.393230,0.480644,-0.024136,...,0.521263,0.426458,-0.053237,0.528025,0.395866,-0.056555,0.533139,0.370218,-0.057810,thankyou


### Now we take the keypoint dataset and model it into the training dataset

In [7]:

# get all unique labels
labels = sorted(df["label"].unique())
print("Labels found:", labels)

# create mapping dictionary
label_to_int = {label: i for i, label in enumerate(labels)}
print("Mapping:", label_to_int)

# add numeric label column
df["label_id"] = df["label"].map(label_to_int)

df.head() # now we have one extra column that is label id


KeyError: 'label'

In [8]:
df.head() 

,filename,bye,hello,no,please,sorry,thankyou,yes
0,0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg....,1,0,0,0,0,0,0
1,0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg....,0,0,0,1,0,0,0
2,0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg....,0,0,0,0,0,0,1
3,0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg....,0,1,0,0,0,0,0
4,0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg....,0,0,0,0,0,1,0


In [10]:
import numpy as np

# Load the file again to be safe
df = pd.read_csv("data/keypoints.csv")

labels = sorted(df["label"].unique())
label_to_int = {label: i for i, label in enumerate(labels)}
df["label_id"] = df["label"].map(label_to_int)

# --- Extract X and y ---
X = df[[f"p{i}" for i in range(63)]].values   # shape: (num_samples, 63)
y = df["label_id"].values                     # shape: (num_samples,)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Example X row:", X[0][:10], "...")
print("Example y:", y[0])



X shape: (60, 63)
y shape: (60,)
Example X row: [ 3.54854554e-01  3.77116144e-01 -2.07906822e-07  3.81328434e-01
  3.60972673e-01  4.68792219e-04  3.99949044e-01  3.35570514e-01
 -8.74609454e-04  4.12803173e-01] ...
Example y: 0


### This means:

You have 60 samples

Each sample has 63 features

Labels are numeric (0–6)

Perfect for training.

In [11]:
##Train/Test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 48
Test size: 12


In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(set(y))  # should be 7

model = keras.Sequential([
    layers.Input(shape=(63,)),          # 63 keypoints
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │         2,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           119 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,695 (10.53 KB)

 Trainable params: 2,695 (10.53 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=8,
    verbose=1
)


Epoch 1/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.2083 - loss: 1.9066 - val_accuracy: 0.2500 - val_loss: 1.8552
Epoch 2/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2083 - loss: 1.8573 - val_accuracy: 0.2500 - val_loss: 1.8233
Epoch 3/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2083 - loss: 1.8314 - val_accuracy: 0.2500 - val_loss: 1.7974
Epoch 4/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2083 - loss: 1.8043 - val_accuracy: 0.2500 - val_loss: 1.7742
Epoch 5/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2083 - loss: 1.7803 - val_accuracy: 0.2500 - val_loss: 1.7609
Epoch 6/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2083 - loss: 1.7631 - val_accuracy: 0.2500 - val_loss: 1.7476
Epoch 7/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.2083 - loss: 1.7490 - val_accuracy: 0.2500 - val_loss: 1.7365
Epoch 8/50
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.2083 - loss: 1.7365 - val_accuracy: 0.2500 - val_loss: 1.7278


In [14]:
model.save("sign_model.h5")
print("Model saved as sign_model.h5")


Model saved as sign_model.h5


In [15]:
from tensorflow import keras

loaded_model = keras.models.load_model("sign_model.h5")
print("Model loaded.")


Model loaded.


In [16]:
import numpy as np
from tensorflow import keras

# Load the saved model
model = keras.models.load_model("sign_model.h5")

# Pick a sample index to test
idx = 0  # you can change this to 1,2,3...

sample = X_test[idx].reshape(1, 63)  # reshape for model
true_label = y_test[idx]

prediction = model.predict(sample)
pred_label = np.argmax(prediction)

print("True label:", true_label)
print("Predicted label:", pred_label)
print("Prediction confidence:", prediction[0][pred_label])


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
True label: 4
Predicted label: 4
Prediction confidence: 0.4668276


In [17]:
import pandas as pd

# load the keypoint dataset
df_kp = pd.read_csv("data/keypoints.csv")

print(df_kp.head())


                                            filename        p0        p1  \
0  0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg....  0.354855  0.377116   
1  0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg....  0.489402  0.658782   
2  0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg....  0.421252  0.410285   
3  0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg....  0.346473  0.296077   
4  0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg....  0.451162  0.550923   

             p2        p3        p4        p5        p6        p7        p8  \
0 -2.079068e-07  0.381328  0.360973  0.000469  0.399949  0.335571 -0.000875   
1  2.008355e-07  0.483480  0.612252 -0.008653  0.494541  0.570583 -0.016955   
2 -2.264598e-07  0.439472  0.411329 -0.007081  0.453712  0.415279 -0.016800   
3  1.623497e-07  0.368758  0.298321 -0.002167  0.387444  0.284682 -0.002193   
4  3.669905e-07  0.414900  0.518674 -0.011077  0.393230  0.480644 -0.024136   

   ...       p54       p55       p56       p57       p58       p59  

In [ ]:

df_kp = pd.read_csv("data/keypoints.csv")
df_kp.head()


,filename,p0,p1,p2,p3,p4,p5,p6,p7,p8,...,p54,p55,p56,p57,p58,p59,p60,p61,p62,label
0,0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg....,0.354855,0.377116,-2.079068e-07,0.381328,0.360973,0.000469,0.399949,0.335571,-0.000875,...,0.364259,0.269544,-0.016077,0.378706,0.279701,-0.014118,0.382285,0.295627,-0.010952,bye
1,0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg....,0.489402,0.658782,2.008355e-07,0.483480,0.612252,-0.008653,0.494541,0.570583,-0.016955,...,0.601380,0.632761,-0.038638,0.619705,0.621701,-0.039907,0.635699,0.613275,-0.039993,please
2,0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg....,0.421252,0.410285,-2.264598e-07,0.439472,0.411329,-0.007081,0.453712,0.415279,-0.016800,...,0.418672,0.396326,-0.025393,0.420461,0.409233,-0.022781,0.420964,0.414627,-0.019283,yes
3,0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg....,0.346473,0.296077,1.623497e-07,0.368758,0.298321,-0.002167,0.387444,0.284682,-0.002193,...,0.398149,0.219555,-0.005620,0.410957,0.209384,-0.004367,0.421394,0.201722,-0.003379,hello
4,0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg....,0.451162,0.550923,3.669905e-07,0.414900,0.518674,-0.011077,0.393230,0.480644,-0.024136,...,0.521263,0.426458,-0.053237,0.528025,0.395866,-0.056555,0.533139,0.370218,-0.057810,thankyou


In [19]:
labels = sorted(df_kp["label"].unique())
label_to_int = {label: i for i, label in enumerate(labels)}
int_to_label = {v: k for k, v in label_to_int.items()}

print("Mapping:", label_to_int)


Mapping: {'bye': 0, 'hello': 1, 'no': 2, 'please': 3, 'sorry': 4, 'thankyou': 5, 'yes': 6}


In [20]:
df_kp["label_id"] = df_kp["label"].map(label_to_int)
df_kp.head()


,filename,p0,p1,p2,p3,p4,p5,p6,p7,p8,...,p55,p56,p57,p58,p59,p60,p61,p62,label,label_id
0,0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg....,0.354855,0.377116,-2.079068e-07,0.381328,0.360973,0.000469,0.399949,0.335571,-0.000875,...,0.269544,-0.016077,0.378706,0.279701,-0.014118,0.382285,0.295627,-0.010952,bye,0
1,0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg....,0.489402,0.658782,2.008355e-07,0.483480,0.612252,-0.008653,0.494541,0.570583,-0.016955,...,0.632761,-0.038638,0.619705,0.621701,-0.039907,0.635699,0.613275,-0.039993,please,3
2,0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg....,0.421252,0.410285,-2.264598e-07,0.439472,0.411329,-0.007081,0.453712,0.415279,-0.016800,...,0.396326,-0.025393,0.420461,0.409233,-0.022781,0.420964,0.414627,-0.019283,yes,6
3,0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg....,0.346473,0.296077,1.623497e-07,0.368758,0.298321,-0.002167,0.387444,0.284682,-0.002193,...,0.219555,-0.005620,0.410957,0.209384,-0.004367,0.421394,0.201722,-0.003379,hello,1
4,0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg....,0.451162,0.550923,3.669905e-07,0.414900,0.518674,-0.011077,0.393230,0.480644,-0.024136,...,0.426458,-0.053237,0.528025,0.395866,-0.056555,0.533139,0.370218,-0.057810,thankyou,5


In [21]:
df_kp.to_csv("data/keypoints.csv", index=False)
print("updated keypoints.csv saved")


updated keypoints.csv saved


In [22]:
df_kp = pd.read_csv("data/keypoints.csv")
df_kp.head()


,filename,p0,p1,p2,p3,p4,p5,p6,p7,p8,...,p55,p56,p57,p58,p59,p60,p61,p62,label,label_id
0,0014-2e0a80d8-4d97-4d79-b240-148b3664681a_jpg....,0.354855,0.377116,-2.079068e-07,0.381328,0.360973,0.000469,0.399949,0.335571,-0.000875,...,0.269544,-0.016077,0.378706,0.279701,-0.014118,0.382285,0.295627,-0.010952,bye,0
1,0010-8ff274ae-29a0-48fb-a6e0-10500b02f9e5_jpg....,0.489402,0.658782,2.008355e-07,0.483480,0.612252,-0.008653,0.494541,0.570583,-0.016955,...,0.632761,-0.038638,0.619705,0.621701,-0.039907,0.635699,0.613275,-0.039993,please,3
2,0020-7ebab096-5a89-4cb6-8d48-0ad7c7025815_jpg....,0.421252,0.410285,-2.264598e-07,0.439472,0.411329,-0.007081,0.453712,0.415279,-0.016800,...,0.396326,-0.025393,0.420461,0.409233,-0.022781,0.420964,0.414627,-0.019283,yes,6
3,0005-ee442e2a-bd99-4ecf-87eb-e83c92111940_jpg....,0.346473,0.296077,1.623497e-07,0.368758,0.298321,-0.002167,0.387444,0.284682,-0.002193,...,0.219555,-0.005620,0.410957,0.209384,-0.004367,0.421394,0.201722,-0.003379,hello,1
4,0014-ee2789d5-86f3-405e-9231-e6386c72e720_jpg....,0.451162,0.550923,3.669905e-07,0.414900,0.518674,-0.011077,0.393230,0.480644,-0.024136,...,0.426458,-0.053237,0.528025,0.395866,-0.056555,0.533139,0.370218,-0.057810,thankyou,5


In [3]:
import cv2
import mediapipe as mp
import numpy as np
from tensorflow import keras
import pandas as pd

# Load keypoint dataset (for label maps)
df_kp = pd.read_csv("data/keypoints.csv")

labels = sorted(df_kp["label"].unique())
label_to_int = {label: i for i, label in enumerate(labels)}
int_to_label = {v: k for k, v in label_to_int.items()}

# Load your trained model
model = keras.models.load_model("sign_model.h5")

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1)

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = hands.process(rgb)

    if result.multi_hand_landmarks:
        for hand_landmarks in result.multi_hand_landmarks:
            mp_draw.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

            keypoints = []
            for lm in hand_landmarks.landmark:
                keypoints.extend([lm.x, lm.y, lm.z])

            keypoints = np.array(keypoints).reshape(1, 63)

            pred = model.predict(keypoints, verbose=0)
            label_id = np.argmax(pred)
            label_name = int_to_label[label_id]
            confidence = pred[0][label_id]

            cv2.putText(
                frame,
                f"{label_name.upper()} ({confidence:.2f})",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 255, 0),
                2
            )

    cv2.imshow("Sign Language Detector", frame)

    key = cv2.waitKey(1)

    if key & 0xFF == ord('q'):  # Q to exit
        break
    if key == 27:              # ESC to exit
        break
    if cv2.getWindowProperty("Sign Language Detector", cv2.WND_PROP_VISIBLE) < 1:
        break

cap.release()
cv2.destroyAllWindows()


d:\sujal\dev\Machine learning projects\sign lang detector\venv\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
